# Question 1.2.5 — exploration d'événements rares par splitting

Ce notebook remplace l'exploration principalement brute-force de `FindingEvRare.ipynb` par une exploration structurée de l'événement rare

$$
A_h=\{N^{+1}_\tau=0,\;N^{+2}_\tau\le h\},
$$

où $\tau$ est le premier instant auquel l'une des deux premières limites se vide. Dans le code :

- `q_bid` correspond à $N^{+1}$ ;
- `q_ask` correspond à $N^{-1}$ ;
- `q2_0` est la taille initiale de la deuxième limite ;
- `plus2_values` contient les valeurs simulées de $N^{+2}_\tau$.

Les quantités principales sont

$$
\mathbb{P}(N^{+2}_\tau \le h \mid N^{+1}_\tau=0)
\quad\text{et}\quad
\mathbb{P}(N^{+1}_\tau=0,\;N^{+2}_\tau\le h).
$$

L'idée est de garder Monte Carlo naïf seulement comme contrôle de cohérence, puis d'utiliser la méthode de splitting déjà implémentée dans `helpers_Splitting.py`.

In [ ]:
import math
import os
import sys
import types
import contextlib
import io
from pathlib import Path
from statistics import NormalDist

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------------------------------------------------
# Robust imports.
# This works both if the helper files are in a helpers/ package
# and if the files helpers_QRNIID.py, helpers_QR2LIM.py,
# helpers_Splitting.py are placed next to this notebook.
# -------------------------------------------------------------------
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if not (ROOT / "helpers").is_dir():
    # Create a virtual package "helpers" whose path is the current folder.
    # This lets `from helpers.helpers_QRNIID import *` work even when
    # helpers_QRNIID.py is next to the notebook.
    helpers_pkg = types.ModuleType("helpers")
    helpers_pkg.__path__ = [str(ROOT)]
    sys.modules.setdefault("helpers", helpers_pkg)

from helpers.helpers_QRNIID import *
from helpers.helpers_QR2LIM import *
from helpers.helpers_Splitting import *

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True

GLOBAL_SEED = 42
rng = np.random.default_rng(GLOBAL_SEED)

## 1. Paramètres de base

On reprend les paramètres de base des simulations précédentes. La deuxième limite réactive reçoit des impulsions d'intensité d'ajout lorsque des retraits se produisent sur la première limite du même côté.

In [ ]:
# Baseline parameters
q0 = 10

mu_plus = 1.0
mu_minus = 1.1
alpha = 0.50
beta = 0.50
gamma_cross = 1.0
lambda_floor = 1e-8

q2_0 = q0
lambda_second_add = mu_plus
lambda_second_remove = theoretical_stationary_lambda_minus(
    mu_plus, mu_minus, alpha, beta
)

b_second = beta
a_second_default = 0.75

# Splitting levels: the event "N^{+1} reaches 0 before N^{-1}"
# is decomposed into progressively lower bid-queue levels.
levels = [8, 6, 4, 2, 0]
levels_alt = [9, 7, 5, 3, 1, 0]

# Moderate default. Increase only in the high-precision block below.
N_PARTICLES = 5_000
T_MAX = 10_000.0

params = {
    "q0": q0,
    "q2_0": q2_0,
    "mu_plus": mu_plus,
    "mu_minus": mu_minus,
    "alpha": alpha,
    "beta": beta,
    "gamma_cross": gamma_cross,
    "lambda_second_add": lambda_second_add,
    "lambda_second_remove": lambda_second_remove,
    "b_second": b_second,
    "lambda_floor": lambda_floor,
}
pd.Series(params)

## 2. Contrôle Monte Carlo naïf

Le Monte Carlo naïf simule directement les deux premières limites jusqu'à $\tau$, puis simule la deuxième limite jusqu'à ce même horizon. Il reste utile comme **sanity check**, mais devient rapidement instable pour une probabilité autour de $10^{-4}$ ou plus petite : l'estimateur peut rester nul simplement parce que l'on n'a observé aucun hit dans l'échantillon.

On le garde donc avec une taille modérée, pour vérifier les ordres de grandeur et la forme qualitative de la distribution conditionnelle.

In [ ]:
def naive_reactive_sanity_check(
    n_samples=3_000,
    a_second=a_second_default,
    h_values=None,
    seed=123,
):
    if h_values is None:
        h_values = np.arange(0, q2_0 + 1)

    local_rng = np.random.default_rng(seed)

    out = sample_second_limit_conditionals_reactive(
        n_samples=n_samples,
        q1_0=q0,
        q2_0=q2_0,
        mu_plus=mu_plus,
        mu_minus=mu_minus,
        alpha=alpha,
        beta=beta,
        lambda_second_add=lambda_second_add,
        lambda_second_remove=lambda_second_remove,
        a_second=a_second,
        b_second=b_second,
        gamma_cross=gamma_cross,
        rng=local_rng,
        t_max=T_MAX,
    )

    first_empty = out["first_empty"]
    hit_zero = out["hit_zero"]
    plus2_at_tau = out["plus2_at_tau"]

    plus1_empty_mask = hit_zero & (first_empty == "plus1")
    plus2_given_plus1_empty = plus2_at_tau[plus1_empty_mask]

    rows = []
    for h in h_values:
        if len(plus2_given_plus1_empty) == 0:
            p_cond = np.nan
            p_joint = 0.0
        else:
            p_cond = np.mean(plus2_given_plus1_empty <= h)
            p_joint = np.mean(plus1_empty_mask & (plus2_at_tau <= h))

        rows.append({
            "h": int(h),
            "n_samples": n_samples,
            "n_plus1_empty": int(np.sum(plus1_empty_mask)),
            "p_hit_plus1_zero_naive": float(np.mean(plus1_empty_mask)),
            "p_cond_second_low_naive": float(p_cond) if not np.isnan(p_cond) else np.nan,
            "p_joint_naive": float(p_joint),
            "a_second": a_second,
        })

    return pd.DataFrame(rows), plus2_given_plus1_empty, out

h_values = np.arange(0, q2_0 + 1)
df_naive, naive_plus2_given_plus1, naive_raw = naive_reactive_sanity_check(
    n_samples=3_000,
    a_second=a_second_default,
    h_values=h_values,
    seed=101,
)

df_naive

In [ ]:
print(f"Nombre de trajectoires conditionnelles N^{{+1}}_tau=0 : {len(naive_plus2_given_plus1)}")
print("Résumé de N^{+2}_tau | N^{+1}_tau=0 :")
pd.Series(naive_plus2_given_plus1).describe()

In [ ]:
if len(naive_plus2_given_plus1) > 0:
    bins = np.arange(
        max(0, np.nanmin(naive_plus2_given_plus1)) - 0.5,
        np.nanmax(naive_plus2_given_plus1) + 1.5,
        1.0,
    )

    plt.figure()
    plt.hist(naive_plus2_given_plus1, bins=bins, density=True, alpha=0.7)
    plt.axvline(q2_0, linestyle="--", linewidth=1, label="taille initiale q2_0")
    plt.yscale("log")
    plt.xlabel(r"$N^{+2}_\tau$")
    plt.ylabel("Densité empirique")
    plt.title(r"Monte Carlo naïf : distribution de $N^{+2}_\tau \mid N^{+1}_\tau=0$")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 3. Estimateur par splitting

La fonction `splitting_estimator` utilise les niveaux intermédiaires pour forcer progressivement les trajectoires vers l'événement

$$
N^{+1}\ \text{atteint }0\ \text{avant }N^{-1}.
$$

À chaque niveau, les trajectoires qui atteignent le niveau demandé sont conservées, les autres sont éliminées, puis les survivantes sont rééchantillonnées. La probabilité d'atteinte de $N^{+1}=0$ est estimée comme produit des fractions de survie. Ensuite, pour les particules finales, on simule $N^{+2}_\tau$ et on estime la probabilité conditionnelle du seuil $h$.

Important : l'implémentation actuelle est orientée vers l'événement “le bid, donc $N^{+1}$, se vide avant l'ask”. En effet, `simulate_until_bid_level` considère l'atteinte d'un niveau bas du bid comme succès et l'épuisement de l'ask comme échec.

In [ ]:
def run_splitting_estimator(
    h,
    a_second=a_second_default,
    n_particles=N_PARTICLES,
    levels=levels,
    seed=202,
    verbose=False,
):
    """Small wrapper around the provided splitting_estimator."""
    local_rng = np.random.default_rng(seed)

    kwargs = dict(
        h=int(h),
        n_particles=int(n_particles),
        levels=list(levels),
        q0=q0,
        q2_0=q2_0,
        mu_plus=mu_plus,
        mu_minus=mu_minus,
        alpha=alpha,
        beta=beta,
        lambda_second_add=lambda_second_add,
        lambda_second_remove=lambda_second_remove,
        a_second=a_second,
        b_second=b_second,
        gamma_cross=gamma_cross,
        lambda_floor=lambda_floor,
        t_max=T_MAX,
        rng=local_rng,
    )

    if verbose:
        return splitting_estimator(**kwargs)

    # The helper prints one line per level; silence it for large loops.
    with contextlib.redirect_stdout(io.StringIO()):
        return splitting_estimator(**kwargs)


h_demo = q2_0
res_demo = run_splitting_estimator(
    h=h_demo,
    a_second=a_second_default,
    n_particles=N_PARTICLES,
    levels=levels,
    seed=202,
    verbose=True,
)

summary_demo = {
    "h": h_demo,
    "a_second": a_second_default,
    "n_particles": N_PARTICLES,
    "p_hit_plus1_zero": res_demo["p_hit_plus1_zero"],
    "p_cond_second_low": res_demo["p_cond_second_low"],
    "p_joint": res_demo["p_joint"],
    "survival_fractions": res_demo["survival_fractions"],
}
summary_demo

### Comment lire les niveaux ?

Avec `levels = [8, 6, 4, 2, 0]`, on part de $q_0=10$ et on demande successivement aux particules d'atteindre $8$, puis $6$, puis $4$, puis $2$, puis $0$, sans que $N^{-1}$ se vide avant. Ces niveaux sont raisonnables si les fractions de survie ne sont ni trop petites ni trop proches de 1.

- Si une fraction est très faible, le niveau est trop brutal.
- Si toutes les fractions sont très proches de 1, on pourrait utiliser moins de niveaux.
- `levels_alt = [9, 7, 5, 3, 1, 0]` est plus progressif, mais coûte un peu plus cher.

In [ ]:
def compare_level_sets(level_sets, h=q2_0, a_second=a_second_default, n_particles=2_000, seed=777):
    rows = []
    for name, lev in level_sets.items():
        res = run_splitting_estimator(
            h=h,
            a_second=a_second,
            n_particles=n_particles,
            levels=lev,
            seed=seed,
            verbose=False,
        )
        rows.append({
            "levels_name": name,
            "levels": lev,
            "p_hit_plus1_zero": res["p_hit_plus1_zero"],
            "p_cond_second_low": res["p_cond_second_low"],
            "p_joint": res["p_joint"],
            "survival_fractions": res["survival_fractions"],
        })
    return pd.DataFrame(rows)

df_levels = compare_level_sets(
    {"baseline": levels, "alternative": levels_alt},
    h=q2_0,
    a_second=a_second_default,
    n_particles=2_000,
    seed=303,
)
df_levels

## 4. Recherche sur le seuil $h$

On étudie la courbe

$$
h \mapsto \mathbb{P}(N^{+2}_\tau \le h \mid N^{+1}_\tau=0).
$$

Remarque d'implémentation : les niveaux de splitting ne dépendent pas de $h$. Il est donc plus efficace de faire une seule simulation de splitting pour un `a_second` donné, puis d'évaluer tous les seuils $h$ sur les mêmes `plus2_values`. Le code ci-dessous permet aussi de forcer des runs indépendants pour chaque $h$ avec `reuse_one_splitting=False`, mais ce n'est pas le choix par défaut.

In [ ]:
def h_grid_from_splitting_result(res, h_values, n_particles, a_second):
    plus2_values = np.asarray(res["plus2_values"])
    p_hit = float(res["p_hit_plus1_zero"])

    rows = []
    for h in h_values:
        p_cond = float(np.mean(plus2_values <= h)) if len(plus2_values) > 0 else np.nan
        p_joint = p_hit * p_cond if not np.isnan(p_cond) else np.nan

        rows.append({
            "h": int(h),
            "p_hit_plus1_zero": p_hit,
            "p_cond_second_low": p_cond,
            "p_joint": p_joint,
            "survival_fractions": list(res["survival_fractions"]),
            "n_particles": int(n_particles),
            "a_second": float(a_second),
            "n_low_final_particles": int(np.sum(plus2_values <= h)),
        })

    return pd.DataFrame(rows)


def search_over_h(
    h_values,
    a_second=a_second_default,
    n_particles=N_PARTICLES,
    levels=levels,
    seed=404,
    reuse_one_splitting=True,
):
    h_values = np.asarray(h_values, dtype=int)

    if reuse_one_splitting:
        res = run_splitting_estimator(
            h=int(np.max(h_values)),
            a_second=a_second,
            n_particles=n_particles,
            levels=levels,
            seed=seed,
            verbose=False,
        )
        df = h_grid_from_splitting_result(res, h_values, n_particles, a_second)
        return df, res

    rows = []
    last_res = None
    for j, h in enumerate(h_values):
        last_res = run_splitting_estimator(
            h=int(h),
            a_second=a_second,
            n_particles=n_particles,
            levels=levels,
            seed=seed + j,
            verbose=False,
        )
        rows.append({
            "h": int(h),
            "p_hit_plus1_zero": float(last_res["p_hit_plus1_zero"]),
            "p_cond_second_low": float(last_res["p_cond_second_low"]),
            "p_joint": float(last_res["p_joint"]),
            "survival_fractions": list(last_res["survival_fractions"]),
            "n_particles": int(n_particles),
            "a_second": float(a_second),
            "n_low_final_particles": int(np.sum(np.asarray(last_res["plus2_values"]) <= h)),
        })

    return pd.DataFrame(rows), last_res


h_values = np.arange(0, q2_0 + 1)

df_h, res_h_baseline = search_over_h(
    h_values=h_values,
    a_second=a_second_default,
    n_particles=N_PARTICLES,
    levels=levels,
    seed=404,
    reuse_one_splitting=True,
)

df_h

In [ ]:
# Smallest h with estimated conditional probability below 1e-4.
# If the estimate is exactly zero, interpret it carefully:
# it means no final splitting particle satisfied N^{+2}_tau <= h.
rare_candidates = df_h[
    (df_h["p_cond_second_low"] < 1e-4)
].copy()

if len(rare_candidates) == 0:
    print("No h in the tested grid has estimated conditional probability below 1e-4.")
else:
    h_star = int(rare_candidates.iloc[0]["h"])
    row_star = rare_candidates.iloc[0]
    print(f"Smallest h with estimated p_cond_second_low < 1e-4: h = {h_star}")
    print(row_star[["h", "p_cond_second_low", "p_joint", "n_low_final_particles"]])

zero_rows = df_h[df_h["n_low_final_particles"] == 0]
if len(zero_rows) > 0:
    print("\nWarning: some estimates are exactly zero because no final particle crossed the threshold.")
    print("These rows should be treated as 'below resolution', not as mathematical zeros.")

In [ ]:
plt.figure()
plot_df = df_h.replace({"p_cond_second_low": {0.0: np.nan}})
plt.plot(plot_df["h"], plot_df["p_cond_second_low"], marker="o")
plt.yscale("log")
plt.xlabel(r"Seuil $h$")
plt.ylabel(r"$\widehat{\mathbb{P}}(N^{+2}_\tau\le h\mid N^{+1}_\tau=0)$")
plt.title(r"Queue basse conditionnelle de $N^{+2}_\tau$ — splitting")
plt.grid(True, which="both", alpha=0.4)
plt.tight_layout()
plt.show()

plt.figure()
plot_df = df_h.replace({"p_joint": {0.0: np.nan}})
plt.plot(plot_df["h"], plot_df["p_joint"], marker="o")
plt.yscale("log")
plt.xlabel(r"Seuil $h$")
plt.ylabel(r"$\widehat{\mathbb{P}}(N^{+1}_\tau=0,\;N^{+2}_\tau\le h)$")
plt.title(r"Probabilité jointe de l'événement rare — splitting")
plt.grid(True, which="both", alpha=0.4)
plt.tight_layout()
plt.show()

## 5. Sensibilité au paramètre `a_second`

On répète la recherche pour plusieurs valeurs de `a_second`.

Interprétation attendue : les retraits sur $N^{+1}$ augmentent l'intensité d'ajout de $N^{+2}$. Donc, quand `a_second` augmente, $N^{+2}_\tau$ a tendance à être plus grand, et l'événement $N^{+2}_\tau \le h$ devient moins probable.

In [ ]:
a_second_values = [0.25, 0.50, 0.75, 1.00, 1.25]

all_sensitivity = []
splitting_outputs_by_a = {}

for k, a_val in enumerate(a_second_values):
    df_tmp, res_tmp = search_over_h(
        h_values=h_values,
        a_second=a_val,
        n_particles=N_PARTICLES,
        levels=levels,
        seed=500 + 100 * k,
        reuse_one_splitting=True,
    )
    all_sensitivity.append(df_tmp)
    splitting_outputs_by_a[a_val] = res_tmp

df_sensitivity = pd.concat(all_sensitivity, ignore_index=True)
df_sensitivity.head()

In [ ]:
plt.figure()
for a_val, grp in df_sensitivity.groupby("a_second"):
    y = grp["p_cond_second_low"].replace(0.0, np.nan)
    plt.plot(grp["h"], y, marker="o", label=f"a_second = {a_val:.2f}")

plt.yscale("log")
plt.xlabel(r"Seuil $h$")
plt.ylabel(r"$\widehat{\mathbb{P}}(N^{+2}_\tau\le h\mid N^{+1}_\tau=0)$")
plt.title(r"Sensibilité de la queue basse à $a_{second}$")
plt.grid(True, which="both", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure()
for a_val, grp in df_sensitivity.groupby("a_second"):
    y = grp["p_joint"].replace(0.0, np.nan)
    plt.plot(grp["h"], y, marker="o", label=f"a_second = {a_val:.2f}")

plt.yscale("log")
plt.xlabel(r"Seuil $h$")
plt.ylabel(r"$\widehat{\mathbb{P}}(N^{+1}_\tau=0,\;N^{+2}_\tau\le h)$")
plt.title(r"Sensibilité de la probabilité jointe à $a_{second}$")
plt.grid(True, which="both", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Histogram of final splitting particles for representative cases.
representative_a_values = [0.25, 0.75, 1.25]

for a_val in representative_a_values:
    plus2_values = np.asarray(splitting_outputs_by_a[a_val]["plus2_values"])
    if len(plus2_values) == 0:
        continue

    plt.figure()
    bins = np.arange(max(0, plus2_values.min()) - 0.5, plus2_values.max() + 1.5, 1.0)
    plt.hist(plus2_values, bins=bins, density=True, alpha=0.7)
    plt.axvline(q2_0, linestyle="--", linewidth=1, label="taille initiale q2_0")
    plt.yscale("log")
    plt.xlabel(r"$N^{+2}_\tau$")
    plt.ylabel("Densité empirique")
    plt.title(rf"Particules finales du splitting — $a_{{second}}={a_val}$")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 6. Incertitude : répétitions indépendantes

Une seule exécution de splitting donne une estimation, mais l'incertitude doit être évaluée par des runs indépendants avec des graines différentes. On répète l'estimateur pour quelques couples $(a_{second},h)$ et on calcule :

- la moyenne empirique ;
- l'écart-type entre runs ;
- un intervalle de confiance empirique à 95 % de type $\bar x \pm 1.96\,s/\sqrt{R}$.

In [ ]:
def repeated_splitting_runs(
    selected_pairs,
    n_runs=8,
    n_particles=3_000,
    levels=levels,
    base_seed=10_000,
):
    rows = []

    for pair_id, (a_val, h_val) in enumerate(selected_pairs):
        for r in range(n_runs):
            seed = base_seed + 1_000 * pair_id + r

            res = run_splitting_estimator(
                h=h_val,
                a_second=a_val,
                n_particles=n_particles,
                levels=levels,
                seed=seed,
                verbose=False,
            )

            rows.append({
                "a_second": float(a_val),
                "h": int(h_val),
                "run": int(r),
                "seed": int(seed),
                "n_particles": int(n_particles),
                "p_hit_plus1_zero": float(res["p_hit_plus1_zero"]),
                "p_cond_second_low": float(res["p_cond_second_low"]),
                "p_joint": float(res["p_joint"]),
                "survival_fractions": list(res["survival_fractions"]),
                "n_low_final_particles": int(np.sum(np.asarray(res["plus2_values"]) <= h_val)),
            })

    return pd.DataFrame(rows)


def summarize_repeated_runs(df_runs):
    rows = []
    z = 1.96

    for (a_val, h_val), grp in df_runs.groupby(["a_second", "h"]):
        row = {
            "a_second": a_val,
            "h": h_val,
            "n_runs": len(grp),
            "n_particles": int(grp["n_particles"].iloc[0]),
        }

        for col in ["p_cond_second_low", "p_joint"]:
            values = grp[col].to_numpy(dtype=float)
            mean = np.mean(values)
            sd = np.std(values, ddof=1) if len(values) > 1 else np.nan
            half_width = z * sd / math.sqrt(len(values)) if len(values) > 1 else np.nan

            row[f"{col}_mean"] = mean
            row[f"{col}_std"] = sd
            row[f"{col}_ci95_low"] = mean - half_width if len(values) > 1 else np.nan
            row[f"{col}_ci95_high"] = mean + half_width if len(values) > 1 else np.nan

        rows.append(row)

    return pd.DataFrame(rows)


selected_pairs = [
    (0.75, 2),
    (0.75, 4),
    (1.25, 4),
]

df_runs = repeated_splitting_runs(
    selected_pairs=selected_pairs,
    n_runs=8,
    n_particles=3_000,
    levels=levels,
    base_seed=12_345,
)

df_uncertainty = summarize_repeated_runs(df_runs)
df_uncertainty

### Option haute précision

Cette cellule est désactivée par défaut. Elle sert à relancer les estimations avec davantage de particules et plus de répétitions, seulement quand les sections précédentes ont identifié les seuils intéressants.

In [ ]:
RUN_HIGH_PRECISION = False

if RUN_HIGH_PRECISION:
    high_precision_pairs = selected_pairs

    df_runs_hp = repeated_splitting_runs(
        selected_pairs=high_precision_pairs,
        n_runs=20,
        n_particles=20_000,
        levels=levels,
        base_seed=50_000,
    )

    df_uncertainty_hp = summarize_repeated_runs(df_runs_hp)
    display(df_uncertainty_hp)

## 7. Paragraphe d'interprétation pour le rapport

> Pour estimer l'événement rare $A_h=\{N^{+1}_\tau=0,\;N^{+2}_\tau\le h\}$, nous utilisons une méthode de splitting multi-niveaux. L'idée est de décomposer l'épuisement de la première limite $N^{+1}$ en plusieurs niveaux intermédiaires, par exemple $8,6,4,2,0$, puis de rééchantillonner les trajectoires qui atteignent successivement ces niveaux avant que la limite opposée $N^{-1}$ ne se vide. Une fois les particules conditionnées sur l'événement $N^{+1}_\tau=0$ obtenues, on simule la deuxième limite réactive $N^{+2}$ jusqu'au temps d'arrêt $\tau$, en tenant compte des impulsions d'ajout produites par les retraits sur $N^{+1}$. Cette approche réduit la variance par rapport à un Monte Carlo naïf, car elle concentre l'effort de simulation sur les trajectoires pertinentes pour l'événement rare. La comparaison avec Monte Carlo naïf reste utile comme contrôle de cohérence, mais seulement dans les régimes où celui-ci produit suffisamment d'occurrences conditionnelles pour être interprétable.

## 8. Résumé des changements par rapport à l'approche brute-force

- Le Monte Carlo naïf est conservé uniquement comme contrôle de cohérence.
- L'estimation principale utilise `splitting_estimator`, qui cible explicitement l'événement $N^{+1}$ atteint zéro avant $N^{-1}$.
- Les probabilités conditionnelles et jointes sont estimées pour une grille complète de seuils $h$.
- La sensibilité à `a_second` est étudiée de façon systématique.
- L'incertitude est estimée par répétitions indépendantes, et non à partir d'un seul run.
- Les figures utilisent une échelle logarithmique pour rendre visibles les probabilités rares.
- Les estimations nulles sont signalées comme limites de résolution numérique, pas comme probabilités exactement nulles.